# 04 — Feature Engineering
This notebook creates useful transaction and customer features from the raw columns.

## Learning objectives
- Create LineValue and calendar features
- Build customer summaries with `groupby().agg()`
- Understand conditional segmentation with `np.where()`

In [1]:
import numpy as np
from retail_analysis.data_loader import load_retail_data
from retail_analysis.cleaning import clean_transactions, split_transaction_sets
from retail_analysis.features import add_transaction_features, create_customer_features

cleaned, _ = clean_transactions(load_retail_data())
sales, _, _ = split_transaction_sets(cleaned)
sales = add_transaction_features(sales)
sales[['Quantity', 'UnitPrice', 'LineValue', 'YearMonth', 'DayOfWeek', 'Hour']].head()

,Quantity,UnitPrice,LineValue,YearMonth,DayOfWeek,Hour
0,6,2.55,15.30,2010-12,Wednesday,8
1,6,3.39,20.34,2010-12,Wednesday,8
2,8,2.75,22.00,2010-12,Wednesday,8
3,6,3.39,20.34,2010-12,Wednesday,8
4,6,3.39,20.34,2010-12,Wednesday,8


Multiplication between Pandas columns is vectorized, so a row-by-row loop is unnecessary. `YearMonth` includes the year, preventing the same month from different years from being combined.

In [2]:
customers = create_customer_features(sales)
customers.head(10)

,CustomerID,TotalPurchaseValue,OrderCount,AverageOrderValue,LastPurchaseDate,RecencyDays,CustomerSegment
0,14646,280206.02,73,3838.438630,2011-12-08 12:12:00,2,Repeat buyer
1,18102,259657.30,60,4327.621667,2011-12-09 11:50:00,1,Repeat buyer
2,17450,194390.79,46,4225.886739,2011-12-01 13:29:00,9,Repeat buyer
3,16446,168472.50,2,84236.250000,2011-12-09 09:15:00,1,Repeat buyer
4,14911,143711.17,201,714.980945,2011-12-08 15:54:00,2,Repeat buyer
5,12415,124914.53,21,5948.310952,2011-11-15 14:22:00,25,Repeat buyer
6,14156,117210.08,55,2131.092364,2011-11-30 10:54:00,10,Repeat buyer
7,17511,91062.38,31,2937.496129,2011-12-07 10:12:00,3,Repeat buyer
8,16029,80850.84,63,1283.346667,2011-11-01 10:27:00,39,Repeat buyer
9,12346,77183.60,1,77183.600000,2011-01-18 10:01:00,326,One-time buyer


`groupby('CustomerID').agg(...)` combines rows for each customer. `nunique()` counts unique orders rather than invoice lines. The recency reference is one day after the latest eligible sales date. These features are not net of refunds.

In [3]:
# np.where selects a label for each True/False condition without a loop
np.where(customers['OrderCount'].eq(1), 'One-time buyer', 'Repeat buyer')[:10]

array(['Repeat buyer', 'Repeat buyer', 'Repeat buyer', 'Repeat buyer',
       'Repeat buyer', 'Repeat buyer', 'Repeat buyer', 'Repeat buyer',
       'Repeat buyer', 'One-time buyer'], dtype='<U14')

In [4]:
customers.groupby('CustomerSegment').agg(
    CustomerCount=('CustomerID', 'count'),
    AveragePurchaseValue=('TotalPurchaseValue', 'mean')
)

,CustomerCount,AveragePurchaseValue
CustomerSegment,,
One-time buyer,1493,411.245520
Repeat buyer,2845,2907.985706


## Interpretation and practice questions
Anonymous transactions remain in sales totals but are absent from the customer table because they have no CustomerID. AverageOrderValue equals TotalPurchaseValue divided by unique OrderCount.

1. Compare the one-time and repeat customer counts.
2. Verify OrderCount for a customer whose invoice contains several line items.
3. Which customers have the lowest RecencyDays values?